# Module 4: Feature Engineering and Dimensionality Reduction
## Interview Simulation Scenarios

Your Name

---

## Getting Started

- Replace **Your Name** above with your full name
- Automatic 0 if you include your student ID or any other identifying number
- Rename the file to `Your_Name.ipynb` before submitting
- When finished, share your Colab link with **Anyone with the link can view** privileges
- Paste the shared link into the Canvas submission

---

## How to Use This Notebook

This notebook is designed to be accessible to all users, including those navigating with a **screen reader**.

### Screen Reader Navigation Guide

Every code section in this notebook gives you a choice. Before each code cell you will find a navigation landmark with two options:

- **Skip this code cell** — Jumps past the code directly to the output explanation. Use this if you want to focus on understanding results without reading raw code.
- **Go back and read the code cell** — Returns you to the top of the code section if you want to review it.

**Tips for screen reader users:**
- Press **H** to jump between major section headings
- Press **K** or **Tab** to move between links, including the skip and return navigation links
- Press **D** to jump between landmark regions

---

## Learning Objectives

By the end of this assignment you will be able to:

1. Identify and correct data leakage in preprocessing pipelines
2. Apply the correct encoding strategy for categorical variables
3. Choose appropriate imputation methods based on data distribution
4. Select between StandardScaler and RobustScaler based on outlier presence
5. Build a multi-stage feature selection pipeline using filter and wrapper methods
6. Interpret PCA scree plots and cumulative variance curves
7. Distinguish between PCA, LDA, and t-SNE and choose appropriately
8. Engineer domain-driven features to improve model performance under time pressure

---

## Deliverables

| Part | Difficulty | Topic | Time |
| :--- | :--- | :--- | :--- |
| **Part 1** | Warm-up | Scaling, encoding, and imputation fundamentals | 5–10 min each |
| **Part 2** | Intermediate | Feature selection pipelines and PCA interpretation | 10–15 min each |
| **Part 3** | Advanced | Production pipeline design and PCA failure modes | 15–20 min each |
| **Part 4** | Challenge | Timed feature engineering and dimensionality reduction | 20–25 min each |

---

## Strategic Approach: The STAR Method

When tackling interview problems, use the **STAR** framework for your explanations:

- **Situation:** Briefly describe the data problem
- **Task:** Identify which technique is needed and why
- **Action:** Write the code and explain your parameter choices
- **Result:** Interpret the output in plain language

---

## Scenario Overview

**Warm-up (Green):** The Scaling Decision — a dataset with `Age` and `Annual Income` being fed to K-Means. What happens without scaling?

**Intermediate (Yellow):** The Redundancy Filter — 50-feature dataset, 10 features correlated above 0.95. Implement automated removal.

**Advanced (Orange):** PCA for High-Dimensional Noise — genomic-style data with thousands of features, severe overfitting. How do you use PCA? How do you defend the choice?

**Challenge (Red):** The Production Pipeline — a t-SNE visualization shows perfect class separation. Your manager wants to ship it to production. Why is this problematic? What is the alternative?

---

## Setup

Run this cell first to import all necessary libraries. All subsequent cells depend on these imports.

<a name="read-code-1"></a>

**Setup Cell — Import All Libraries**

This cell imports every library used in the notebook: `pandas` and `numpy` for data manipulation, `matplotlib` and `seaborn` for visualization, scikit-learn modules for preprocessing, feature selection, decomposition, and modeling. The random seed is fixed at 42 for reproducibility.

<nav aria-label="Code cell 1 navigation">
<a href="#skip-code-1" aria-label="Skip code cell 1 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import mutual_info_classif, RFE, SelectKBest, f_classif, VarianceThreshold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_breast_cancer, load_wine, load_digits
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Setup complete.')

<a name="skip-code-1"></a>

**Expected output:** `Setup complete.`

If you see an import error, install the missing package with `!pip install <package_name>` and re-run this cell.

<nav aria-label="Return navigation for code cell 1">
<a href="#read-code-1" aria-label="Go back and read code cell 1">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 1: Warm-Up Problems

These problems test fundamentals that appear in nearly every data science interview. Each should take 5 to 10 minutes. Identify the issue, explain *why* it is wrong, and provide the corrected code.

---

## Problem 1.1: Detect the Scaling Mistake

**Difficulty:** Warm-up | **Context:** Debugging data leakage in a preprocessing pipeline

**Scenario:** A junior data scientist at a hospital is building a tumor classification model using the Breast Cancer Wisconsin dataset. They scaled the training data and then incorrectly handled the test data. Examine their code in the cell below.

**Your Tasks:**
1. Identify the mistake in the buggy code
2. Explain *why* it is wrong (frame your answer around data leakage)
3. Write the corrected code

**Interview Soundbite to aim for:**
> 'Calling `fit_transform()` on the test set causes the scaler to learn statistics from test data. This violates the principle that the test set must simulate unseen future data. The fix is to `fit` only on training data and `transform` everything else.'

<a name="read-code-2"></a>

**Buggy Code Cell — Scaling Mistake**

This cell loads the Breast Cancer Wisconsin dataset (569 samples, 30 numerical features) and simulates the mistake of calling `fit_transform()` on the test set. Run it to see what the incorrect pipeline produces. Your job is to identify and fix the error in Problem 1.1 below.

<nav aria-label="Code cell 2 navigation">
<a href="#skip-code-2" aria-label="Skip code cell 2 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Junior data scientist's code:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# ❌ MISTAKE HERE — what is wrong with this line?
X_test_scaled = scaler.fit_transform(X_test)

pca = PCA(n_components=2)
X_test_pca = pca.fit_transform(X_test_scaled)
print('Test data transformed shape:', X_test_pca.shape)

<a name="skip-code-2"></a>

**Expected output:** `Test data transformed shape: (114, 2)`

The code runs without error — which makes the bug easy to miss. The problem is invisible in the output but corrupts the model's learned representation. Now answer the three tasks in the cell below.

<nav aria-label="Return navigation for code cell 2">
<a href="#read-code-2" aria-label="Go back and read code cell 2">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-3"></a>

**Problem 1.1 Answer Cell**

Fill in your answers as comments inside this cell. For the fix, write the corrected code that properly separates `fit` (training only) from `transform` (applied to both sets using training statistics).

<nav aria-label="Code cell 3 navigation">
<a href="#skip-code-3" aria-label="Skip code cell 3 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# 1. What is the mistake?
# ANSWER:


# 2. Why is it wrong? (Reference data leakage and what 'fit' actually does)
# ANSWER:


# 3. Fixed code:
# scaler = StandardScaler()
# X_train_scaled = ???
# X_test_scaled  = ???  # only one word changes here

<a name="skip-code-3"></a>

**Solution summary:**

The mistake is `scaler.fit_transform(X_test)`. When `fit` runs on the test set, the scaler learns the test set's mean and standard deviation. This means the same raw value (e.g., a tumor radius of 15mm) maps to a different scaled value depending on which set it appears in — the model sees inconsistent numerical inputs.

The fix is `scaler.transform(X_test)` — no `fit`. The scaler already learned the statistics from `X_train_scaled = scaler.fit_transform(X_train)`. Those parameters are applied unchanged to the test set.

<nav aria-label="Return navigation for code cell 3">
<a href="#read-code-3" aria-label="Go back and read code cell 3">&#8617; Go back and read the code cell</a>
</nav>

---
## Problem 1.2: The Dummy Variable Trap

**Difficulty:** Warm-up | **Context:** Encoding logic for linear models

**Scenario:** A streaming analytics team is encoding a `Content_Genre` variable (Action, Drama, Comedy, Documentary, Thriller) before training a linear regression model to predict monthly stream counts.

**The Question:** The column has 5 unique genres. How many dummy variables should be created, and why?

**The Short Answer:** Create **4** dummy variables — always N minus 1.

**Why?** If you include all 5 columns, one is perfectly predicted by the others: if a title is not Action, Drama, Comedy, or Documentary, it *must* be Thriller. This perfect multicollinearity makes the design matrix singular — the coefficients cannot be solved uniquely. The dropped category becomes the 'reference baseline' and all other coefficients are interpreted relative to it.

**✏️ Your Task:** Run the code cell below and confirm the column count. Add a comment explaining which genre was dropped and what the remaining coefficients represent.

<a name="read-code-4"></a>

**Cell 4 — Demonstrate the Dummy Variable Trap**

This cell creates a small streaming dataset with a `Content_Genre` column and encodes it two ways: once including all 5 genres (wrong) and once with `drop_first=True` (correct). Compare the column counts in the printed output.

<nav aria-label="Code cell 4 navigation">
<a href="#skip-code-4" aria-label="Skip code cell 4 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
stream_data = pd.DataFrame({
    'Content_Genre': ['Action','Drama','Comedy','Documentary','Thriller',
                      'Action','Drama','Comedy','Documentary','Thriller'],
    'Monthly_Streams_M': [4.2, 3.8, 5.1, 2.7, 3.3, 4.5, 3.6, 5.3, 2.9, 3.1]
})

print(f'Unique genres: {stream_data["Content_Genre"].nunique()}')

# Wrong: all 5 dummy columns included
wrong_way   = pd.get_dummies(stream_data, columns=['Content_Genre'], drop_first=False)
# Correct: N-1 dummy columns
correct_way = pd.get_dummies(stream_data, columns=['Content_Genre'], drop_first=True)

print('\nWrong way genre columns:  ', [c for c in wrong_way.columns   if 'Genre' in c])
print('Correct way genre columns:', [c for c in correct_way.columns if 'Genre' in c])
print('\nCorrect encoded data:')
print(correct_way.head())

# Add your comment below:
# Genre dropped: ???
# What the remaining coefficients represent: ???

<a name="skip-code-4"></a>

**Expected output:**
`Unique genres: 5`
`Wrong way genre columns: ['Content_Genre_Action', 'Content_Genre_Comedy', 'Content_Genre_Documentary', 'Content_Genre_Drama', 'Content_Genre_Thriller']`
`Correct way genre columns: ['Content_Genre_Comedy', 'Content_Genre_Documentary', 'Content_Genre_Drama', 'Content_Genre_Thriller']`

`drop_first=True` drops the alphabetically first category (Action). Each remaining coefficient tells you: 'How many additional monthly streams does this genre earn compared to an Action title?'

<nav aria-label="Return navigation for code cell 4">
<a href="#read-code-4" aria-label="Go back and read code cell 4">&#8617; Go back and read the code cell</a>
</nav>

---
## Problem 1.3: Outlier-Aware Scaling and Imputation

**Difficulty:** Warm-up | **Context:** Scaler and imputer selection based on data distribution

### Part A: Choosing the Right Scaler

**Interview Question:** *'If your dataset contains significant outliers, would you still use StandardScaler? Why or why not?'*

**The Problem with StandardScaler:** It uses the mean (μ) and standard deviation (σ). Both metrics are non-robust — a single extreme value inflates σ and shifts μ away from the data center. The result: all normal values get squeezed into a tiny range while the outlier dominates.

**The Alternative — RobustScaler:**
- Uses the **Median** (unaffected by the extreme value)
- Uses the **IQR** (Interquartile Range: Q3 − Q1, covering only the middle 50% of the data)

### Part B: Choosing the Right Imputer

**The Decision Matrix:**

| Feature Type | Distribution | Best Method | Why |
| :--- | :--- | :--- | :--- |
| Numeric | Normal / Symmetric | **Mean** | Most statistically efficient for Gaussian data |
| Numeric | Skewed or has outliers | **Median** | Robust to long tails and extreme values |
| Categorical | Any | **Mode** | Cannot average text labels |
| Any | Pattern-based missingness | **KNN / MICE** | Uses other features to predict the missing value |

**Interview Soundbite:**
> 'My imputation choice depends on the distribution. For symmetric numeric data I use the mean. For skewed or outlier-heavy data I prefer the median — the mean would be pulled toward the tail and introduce bias. For categorical data I default to the mode or create an explicit Missing category to preserve the signal that the value was absent.'

<a name="read-code-5"></a>

**Cell 5 — Compare StandardScaler vs RobustScaler, and Demonstrate Imputation Choices**

This cell generates three distributions: normal, right-skewed (exponential), and uniform. It injects missing values and an extreme outlier into each. It then applies StandardScaler and RobustScaler to illustrate the difference, and applies the appropriate SimpleImputer to each distribution type.

<nav aria-label="Code cell 5 navigation">
<a href="#skip-code-5" aria-label="Skip code cell 5 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer

# --- Scaler Comparison ---
X_outlier = np.array([10, 12, 11, 13, 12, 11, 1000]).reshape(-1, 1)

ss_result = StandardScaler().fit_transform(X_outlier).flatten()
rs_result = RobustScaler().fit_transform(X_outlier).flatten()

print('=== Scaler Comparison (with extreme outlier at 1000) ===')
print(f'Original values: {X_outlier.flatten()}')
print(f'StandardScaler: {np.round(ss_result, 2)}')
print(f'RobustScaler:   {np.round(rs_result, 2)}')
print('Note: StandardScaler squeezes all normal values together; RobustScaler keeps them spread.')

# --- Imputation Comparison ---
np.random.seed(42)
data_normal  = np.random.normal(50, 10, 100).astype(float)       # Symmetric
data_skewed  = np.random.exponential(scale=20, size=100).astype(float)  # Right-skewed
data_uniform = np.random.uniform(0, 100, 100).astype(float)      # Uniform

# Inject missing values
for d in [data_normal, data_skewed, data_uniform]:
    d[np.random.choice(100, 10, replace=False)] = np.nan

imp_mean   = SimpleImputer(strategy='mean')
imp_median = SimpleImputer(strategy='median')

d1_imputed = imp_mean.fit_transform(data_normal.reshape(-1,1)).flatten()
d2_imputed = imp_median.fit_transform(data_skewed.reshape(-1,1)).flatten()
d3_imputed = imp_median.fit_transform(data_uniform.reshape(-1,1)).flatten()

print('\n=== Imputation Choices ===')
print(f'Normal data  -> Mean imputed. Mean: {d1_imputed.mean():.2f}')
print(f'Skewed data  -> Median imputed. Median: {np.median(d2_imputed):.2f}')
print(f'Uniform data -> Median imputed (safer default). Median: {np.median(d3_imputed):.2f}')

<a name="skip-code-5"></a>

**Expected output:** Two sections. In the scaler section, StandardScaler maps the normal values (10–13) to approximately −0.3 while the outlier (1000) maps to about 2.6 — nearly all variance goes to the outlier. RobustScaler keeps the normal values spread around 0 and pushes the outlier far above.

In the imputation section, mean imputation is appropriate for the symmetric normal distribution. Median imputation is correct for the exponential (skewed) distribution because the mean is pulled rightward by the long tail.

**✏️ Your Task:** Add a comment to this cell explaining which dataset you would use RobustScaler on and why.

<nav aria-label="Return navigation for code cell 5">
<a href="#read-code-5" aria-label="Go back and read code cell 5">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 2: Intermediate Problems

These problems test your ability to build multi-step pipelines and interpret results quantitatively. Each should take 10 to 15 minutes.

---

## Problem 2.1: Mutual Information vs Pearson Correlation

**Difficulty:** Intermediate | **Context:** Feature selection for non-linear relationships

**Scenario:** You are building a predictive model for a clinical trial and notice that a patient's `dosage_level` feature has a perfect **U-shaped (quadratic) relationship** with `response_score`. A colleague filters features using Pearson Correlation and discards `dosage_level` because the score is near zero.

**The Question:** Is your colleague right? What should you use instead?

**Key Insight:**
- **Pearson Correlation** measures *linear* dependency only. A symmetric U-shape produces a correlation of approximately 0 because the positive and negative halves cancel.
- **Mutual Information (MI)** measures *any* statistical dependency — linear, quadratic, or otherwise — using entropy. A perfect quadratic relationship will show a high MI score.

**Interview Soundbite:**
> 'Pearson correlation is a good first pass but is blind to non-linear patterns. For robust feature selection I prefer Mutual Information, which uses entropy to capture any form of statistical dependency between a feature and the target.'

<a name="read-code-6"></a>

**Cell 6 — Demonstrate Correlation vs Mutual Information on Quadratic Data**

This cell generates `dosage_level` values over a symmetric range and computes a quadratic response score with added noise. It calculates Pearson correlation (expected: near 0) and Mutual Information (expected: substantially above 0) for the same feature.

<nav aria-label="Code cell 6 navigation">
<a href="#skip-code-6" aria-label="Skip code cell 6 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Simulate dosage (symmetric range) and quadratic response
np.random.seed(42)
dosage  = np.linspace(-10, 10, 200)
response = dosage**2 + np.random.normal(0, 8, 200)  # Quadratic + noise

# Pearson correlation
corr = np.corrcoef(dosage, response)[0, 1]

# Mutual Information (binarize response for classif MI)
response_binary = (response > response.mean()).astype(int)
mi_score = mutual_info_classif(dosage.reshape(-1, 1), response_binary, random_state=42)[0]

print('=== Correlation vs Mutual Information ===')
print(f'Pearson Correlation:      {corr:.4f}  (near 0 — misses the quadratic pattern)')
print(f'Mutual Information Score: {mi_score:.4f}  (captures non-linear dependency)')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(dosage, response, alpha=0.5, color='steelblue', edgecolors='none')
axes[0].set_title(f'Dosage vs Response\n(Pearson r = {corr:.3f})', fontsize=13)
axes[0].set_xlabel('Dosage Level')
axes[0].set_ylabel('Response Score')
axes[0].grid(alpha=0.3)

axes[1].bar(['Pearson\n|Correlation|', 'Mutual\nInformation'],
           [abs(corr), mi_score], color=['#e74c3c','#27ae60'], alpha=0.8, edgecolor='black')
axes[1].set_title('Feature Relevance Score Comparison', fontsize=13)
axes[1].set_ylabel('Score')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fig_mi_vs_corr.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-6"></a>

**Expected output:** Pearson correlation close to 0 (typically between −0.05 and 0.05). Mutual Information substantially above 0 (typically 0.10 to 0.25). The scatter plot shows a clear U-shape that Pearson misses entirely.

**Accessibility note:** Add a Markdown cell below describing both charts. Example: 'The left scatter plot shows a clear U-shaped relationship between dosage and response, confirming the quadratic pattern. The right bar chart shows the Mutual Information score is much higher than the near-zero Pearson correlation, demonstrating that MI correctly identifies the feature as informative.'

<nav aria-label="Return navigation for code cell 6">
<a href="#read-code-6" aria-label="Go back and read code cell 6">&#8617; Go back and read the code cell</a>
</nav>

## Problem 2.1 (continued): The Three-Stage Feature Selection Pipeline

**Scenario:** The clinical trial dataset now has 50 features. Running RFE on all 50 immediately would be slow and computationally wasteful. Design a tiered approach that moves from fast, broad filters to precise, model-based selection.

**The Strategy:**

| Stage | Method | Goal | Why |
| :--- | :--- | :--- | :--- |
| 1 — Janitor | VarianceThreshold | Remove near-constant features | Zero-variance features carry no information |
| 2 — Filter | SelectKBest (MI) | Rank top 20 by information gain | Fast; captures non-linear relationships |
| 3 — Wrapper | RFE (Logistic Regression) | Find the best interacting set of 10 | Slow but captures feature interactions |

**✏️ Your Task:** Fill in the blanks (marked `???`) in the code cell below.

<a name="read-code-7"></a>

**Cell 7 — Three-Stage Feature Selection Pipeline**

This cell generates a synthetic 50-feature classification dataset representing clinical trial data. It runs three selection stages in sequence. Several lines are incomplete — fill in the `???` placeholders. At the end, cross-validation accuracy is compared between all 50 features and the final 10.

<nav aria-label="Code cell 7 navigation">
<a href="#skip-code-7" aria-label="Skip code cell 7 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.datasets import make_classification
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Generate synthetic clinical trial data: 50 features, informative subset
X, y = make_classification(
    n_samples=400, n_features=50, n_informative=15,
    n_redundant=10, n_repeated=5, random_state=42
)
print(f'Starting feature shape: {X.shape}')

# =================================================================
# STAGE 1: Variance Threshold (The Janitor)
# Goal: Drop features that vary less than 1% — they carry no signal.
# =================================================================
selector_var = VarianceThreshold(threshold=0.01)
X_var = selector_var.fit_transform(X)
print(f'After Stage 1 (Variance): {X_var.shape[1]} features remaining')

# =================================================================
# STAGE 2: SelectKBest using Mutual Information (The Filter)
# Goal: Rank remaining features by MI score; keep top 20.
# =================================================================
selector_k = SelectKBest(score_func=???, k=20)  # YOUR CODE: fill in the scoring function
X_k = selector_k.fit_transform(X_var, y)
print(f'After Stage 2 (KBest MI): {X_k.shape[1]} features remaining')

# =================================================================
# STAGE 3: Recursive Feature Elimination (The Wrapper)
# Goal: Use a model to find the 10 features that work best together.
# =================================================================
estimator    = LogisticRegression(solver='liblinear', random_state=42)
selector_rfe = RFE(estimator=???, n_features_to_select=10)  # YOUR CODE: fill in the estimator
X_final = selector_rfe.fit_transform(X_k, y)
print(f'After Stage 3 (RFE):      {X_final.shape[1]} features remaining')

# =================================================================
# PERFORMANCE COMPARISON
# =================================================================
model     = LogisticRegression(solver='liblinear', random_state=42)
score_all = cross_val_score(model, X, y, cv=5).mean()
score_sel = cross_val_score(model, X_final, y, cv=5).mean()

print('\n' + '-'*40)
print(f'Accuracy (all 50 features):     {score_all:.4f}')
print(f'Accuracy (selected 10 features):{score_sel:.4f}')
print(f'Feature reduction: {X.shape[1]} → {X_final.shape[1]} ({((X.shape[1]-X_final.shape[1])/X.shape[1]*100):.0f}% reduction)')

<a name="skip-code-7"></a>

**Expected output after filling in the blanks:**

`Starting feature shape: (400, 50)`
`After Stage 1 (Variance): ~45 features remaining`
`After Stage 2 (KBest MI): 20 features remaining`
`After Stage 3 (RFE):      10 features remaining`

Accuracy with 10 selected features should be comparable to (or better than) all 50 features. This demonstrates that removing noise and redundancy improves the signal-to-noise ratio.

**Blanks to fill:**
- Stage 2: `score_func=mutual_info_classif`
- Stage 3: `estimator=estimator`

<nav aria-label="Return navigation for code cell 7">
<a href="#read-code-7" aria-label="Go back and read code cell 7">&#8617; Go back and read the code cell</a>
</nav>

---
## Problem 2.2: PCA Scree Plot Analysis

**Difficulty:** Intermediate | **Context:** Interpreting PCA results for stakeholders

**Scenario:** You have applied PCA to the Breast Cancer Wisconsin dataset (30 features). A stakeholder asks: *'How many components should we keep?'*

**Interview Skill Assessed:** Interpreting PCA output and communicating the decision clearly to a non-technical audience.

**Your Tasks:**
1. Using the elbow method, how many components would you choose?
2. How many components explain 90% of the variance? How many for 95%?
3. Write a 2–3 sentence explanation suitable for a non-technical clinical manager.
4. What is the trade-off between using fewer versus more components?

<a name="read-code-8"></a>

**Cell 8 — PCA Scree Plot and Cumulative Variance (Breast Cancer Dataset)**

This cell loads the Breast Cancer Wisconsin dataset (569 samples, 30 features), standardizes it, and fits PCA with all 30 components. It then plots two panels: a scree plot (individual explained variance per component) and a cumulative variance curve with reference lines at 90% and 95%.

<nav aria-label="Code cell 8 navigation">
<a href="#skip-code-8" aria-label="Skip code cell 8 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
cancer = load_breast_cancer()
X_bc = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y_bc = cancer.target

scaler = StandardScaler()
X_bc_scaled = scaler.fit_transform(X_bc)

pca_full = PCA()
pca_full.fit(X_bc_scaled)

explained  = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

n_90 = np.argmax(cumulative >= 0.90) + 1
n_95 = np.argmax(cumulative >= 0.95) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, len(explained)+1), explained, edgecolor='black', color='steelblue', alpha=0.8)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot — Breast Cancer Dataset', fontsize=13)
axes[0].set_xticks(range(1, len(explained)+1, 2))
axes[0].grid(axis='y', alpha=0.3)

# Cumulative variance
axes[1].plot(range(1, len(cumulative)+1), cumulative, marker='o', markersize=4,
            color='steelblue', linewidth=2, label='Cumulative Variance')
axes[1].axhline(0.90, color='orange', linestyle='--', linewidth=2, label=f'90% ({n_90} components)')
axes[1].axhline(0.95, color='red',    linestyle='--', linewidth=2, label=f'95% ({n_95} components)')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Variance — Breast Cancer Dataset', fontsize=13)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig_pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Components for 90% variance: {n_90}')
print(f'Components for 95% variance: {n_95}')

<a name="skip-code-8"></a>

**Expected output:** Two charts and printed component counts. The scree plot shows a steep drop after the first few components and then levels off. The elbow typically appears around components 3 to 5. Approximately 7 to 10 components explain 90% of the variance in this dataset.

**Accessibility note:** Add a Markdown cell below describing both charts. Note the elbow location in the scree plot and the component counts where the cumulative curve crosses the 90% and 95% reference lines.

<nav aria-label="Return navigation for code cell 8">
<a href="#read-code-8" aria-label="Go back and read code cell 8">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-9"></a>

**Cell 9 — Record Your PCA Analysis Answers**

Fill in your written answers as comments in this cell. Your answer for Question 3 should be written in plain language suitable for a clinical manager who does not know what PCA means.

<nav aria-label="Code cell 9 navigation">
<a href="#skip-code-9" aria-label="Skip code cell 9 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# 1. Elbow method component choice:
# ANSWER:

# 2. Components for variance thresholds:
# 90% variance: ??? components
# 95% variance: ??? components

# 3. Explanation for a non-technical clinical manager (2-3 sentences):
# ANSWER:

# 4. Trade-offs between fewer vs more components:
# Fewer (2-3): ???
# More (15+):  ???

<a name="skip-code-9"></a>

**Model answer for Question 3:**
'Instead of analyzing 30 measurements per patient separately, we condensed them into 7 combined scores that still capture 90% of the information. This makes the model simpler and faster without meaningfully sacrificing accuracy. The 10% of information we removed is mostly measurement noise rather than clinically relevant signal.'

**Trade-off summary:**
Fewer components: easier to visualize and faster to compute, but risks discarding real signal. More components: preserves more information but reduces the benefit of dimensionality reduction and makes the model harder to interpret.

<nav aria-label="Return navigation for code cell 9">
<a href="#read-code-9" aria-label="Go back and read code cell 9">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 3: Advanced Problems

These problems simulate take-home and on-site interview challenges where you must write production-quality code and defend design decisions. Each should take 15 to 20 minutes.

---

## Problem 3.1: Complete Preprocessing Pipeline

**Difficulty:** Advanced | **Context:** Production-ready pipeline for messy healthcare data

**Scenario:** You are a data scientist at a hospital network building a patient readmission prediction model. The dataset has mixed numeric and categorical features, missing values, outliers (income and lab values are right-skewed), and features on very different scales.

**Task:** Design and implement a `preprocess_data()` function that:
1. Handles missing values appropriately for each feature type
2. Detects and caps outliers in numeric features using the IQR method
3. Encodes categorical variables correctly (avoiding the dummy variable trap)
4. Scales numeric features using a strategy appropriate for the data
5. Works consistently on both training and test data (no data leakage)

**Key Insight:** When `is_train=True`, *save* all fitted transformers. When `is_train=False`, *reuse* them. This mirrors a real production pipeline.

<a name="read-code-10"></a>

**Cell 10 — Create the Messy Patient Readmission Dataset**

This cell generates a realistic 300-patient dataset with the following features: age, annual income (right-skewed), blood glucose (normal), BMI, length of stay (integer), insurance type (categorical), primary diagnosis category (categorical), and the binary target `readmitted`. Missing values are injected into income, glucose, and insurance. Extreme outliers are added to income. Run this cell before attempting Problem 3.1.

<nav aria-label="Code cell 10 navigation">
<a href="#skip-code-10" aria-label="Skip code cell 10 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
np.random.seed(42)
n_patients = 300

patient_data = pd.DataFrame({
    'age':               np.random.randint(18, 90, n_patients),
    'annual_income':     np.random.exponential(45_000, n_patients),
    'blood_glucose':     np.random.normal(120, 30, n_patients),
    'bmi':               np.random.normal(27, 5, n_patients),
    'length_of_stay':    np.random.randint(1, 30, n_patients),
    'insurance_type':    np.random.choice(['Medicare','Medicaid','Private','Uninsured'], n_patients),
    'diagnosis_category':np.random.choice(['Cardiac','Respiratory','Metabolic','Neurological'], n_patients),
})

# Binary target
patient_data['readmitted'] = (
    (patient_data['blood_glucose'] > 150).astype(int) * 0.35
    + (patient_data['length_of_stay'] > 14).astype(int) * 0.30
    + (patient_data['bmi'] > 35).astype(int) * 0.20
    + np.random.random(n_patients) * 0.15
) > 0.45
patient_data['readmitted'] = patient_data['readmitted'].astype(int)

# Inject missing values
patient_data.loc[np.random.choice(n_patients, 25, replace=False), 'annual_income'] = np.nan
patient_data.loc[np.random.choice(n_patients, 20, replace=False), 'blood_glucose']  = np.nan
patient_data.loc[np.random.choice(n_patients, 15, replace=False), 'insurance_type']  = np.nan

# Inject income outliers
patient_data.loc[np.random.choice(n_patients, 5, replace=False), 'annual_income'] = np.random.uniform(400_000, 900_000, 5)

print('Patient Dataset Info:')
print(patient_data.info())
print('\nMissing Values:')
print(patient_data.isnull().sum())
print('\nFirst 5 rows:')
print(patient_data.head())

<a name="skip-code-10"></a>

**Expected output:** Dataset info showing 300 patients, 8 columns (7 features + target). Missing value counts: approximately 25 in `annual_income`, 20 in `blood_glucose`, 15 in `insurance_type`. Verify these before building your pipeline.

<nav aria-label="Return navigation for code cell 10">
<a href="#read-code-10" aria-label="Go back and read code cell 10">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-11"></a>

**Cell 11 — Implement the Preprocessing Function**

This cell contains the function signature and docstring for `preprocess_data()`. Your task is to implement the body following the structured hints inside. The function must handle training (fit + transform) and test (transform only) modes correctly. Below the function definition, the test harness splits the data and calls the function on both sets.

<nav aria-label="Code cell 11 navigation">
<a href="#skip-code-11" aria-label="Skip code cell 11 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
def preprocess_data(df, is_train=True, artifacts=None):
    """
    Production-ready preprocessing pipeline for patient readmission data.

    Parameters
    ----------
    df : pd.DataFrame — input features (no target column)
    is_train : bool — True = fit + transform; False = transform only
    artifacts : dict or None — fitted transformers from training (required when is_train=False)

    Returns
    -------
    df_processed : pd.DataFrame — clean, fully numeric feature set
    artifacts    : dict — fitted transformers {imputer_num, bounds, scaler, encoder}
    """
    df = df.copy()
    if artifacts is None:
        artifacts = {}

    numeric_cols     = ['age', 'annual_income', 'blood_glucose', 'bmi', 'length_of_stay']
    categorical_cols = ['insurance_type', 'diagnosis_category']

    # -------------------------------------------------------------------
    # STEP 1: Impute missing values
    # Use median for numeric (robust to skew/outliers); mode for categorical
    # -------------------------------------------------------------------
    # YOUR CODE HERE


    # -------------------------------------------------------------------
    # STEP 2: Cap outliers in numeric columns using IQR
    # Fit bounds on training data only; apply to both sets
    # df[col].clip(lower=lower_bound, upper=upper_bound)
    # -------------------------------------------------------------------
    # YOUR CODE HERE


    # -------------------------------------------------------------------
    # STEP 3: Encode categorical variables
    # Use pd.get_dummies with drop_first=True to avoid the dummy trap
    # -------------------------------------------------------------------
    # YOUR CODE HERE


    # -------------------------------------------------------------------
    # STEP 4: Scale numeric features
    # Use RobustScaler (income is right-skewed and has outliers)
    # Fit on training data only
    # -------------------------------------------------------------------
    # YOUR CODE HERE


    return df, artifacts


# --- Test harness ---
X_pat = patient_data.drop('readmitted', axis=1)
y_pat = patient_data['readmitted']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_pat, y_pat, test_size=0.3, random_state=42
)

X_train_proc, artifacts = preprocess_data(X_train_p, is_train=True)
X_test_proc,  _         = preprocess_data(X_test_p,  is_train=False, artifacts=artifacts)

print('Processed training shape:', X_train_proc.shape)
print('Processed test shape:    ', X_test_proc.shape)
print('Remaining nulls (train): ', X_train_proc.isnull().sum().sum())
print('Remaining nulls (test):  ', X_test_proc.isnull().sum().sum())

<a name="skip-code-11"></a>

**Expected output after completing the function:**

`Processed training shape: (210, N)` where N depends on your encoding choices.
`Processed test shape:     (90, N)` — same number of columns.
`Remaining nulls (train): 0`
`Remaining nulls (test):  0`

**Interview talking points to include in your comments:**
- Median imputation chosen because `annual_income` is exponentially distributed
- IQR capping preserves all rows while limiting outlier influence
- Bounds fitted on training data only and reused on test data
- RobustScaler preferred over StandardScaler for skewed clinical variables
- `drop_first=True` avoids perfect multicollinearity (dummy variable trap)

<nav aria-label="Return navigation for code cell 11">
<a href="#read-code-11" aria-label="Go back and read code cell 11">&#8617; Go back and read the code cell</a>
</nav>

---
## Problem 3.2: When PCA Goes Wrong

**Difficulty:** Advanced | **Context:** Choosing between PCA and LDA

**Scenario:** A colleague applied PCA to reduce a two-class medical imaging dataset to 1D before classification. Model accuracy dropped significantly. Diagnose the failure and recommend the correct approach.

**Your Tasks:**
1. Identify *why* PCA hurt model performance in this case
2. Implement the better alternative and compare accuracy
3. State clear guidelines for when to use PCA vs LDA vs t-SNE

**Concept Check:** This question tests whether you understand that PCA maximizes **variance** while LDA maximizes **class separability** — these are fundamentally different objectives.

<a name="read-code-12"></a>

**Cell 12 — PCA Failure Demonstration**

This cell generates a 2D dataset where two classes are well-separated but the primary axis of variance runs diagonally (not aligned with the class boundary). PCA projects onto PC1 (maximum variance), which collapses the two classes together. SVC accuracy before and after PCA is printed. Two scatter plots show the original data and the 1D projection.

<nav aria-label="Code cell 12 navigation">
<a href="#skip-code-12" aria-label="Skip code cell 12 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.datasets import make_classification
from sklearn.svm import SVC

X_sep, y_sep = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, flip_y=0.05, class_sep=3.0, random_state=42
)

pca_1d  = PCA(n_components=1)
X_pca1d = pca_1d.fit_transform(X_sep)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls, label in zip([0, 1], ['Class 0 (No Readmission)', 'Class 1 (Readmitted)']):
    axes[0].scatter(X_sep[y_sep==cls, 0], X_sep[y_sep==cls, 1],
                   label=label, alpha=0.65, edgecolors='none')
axes[0].set_title('Original Data (2D)'); axes[0].set_xlabel('Feature 1'); axes[0].set_ylabel('Feature 2')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for cls, label in zip([0, 1], ['Class 0', 'Class 1']):
    axes[1].scatter(X_pca1d[y_sep==cls], np.zeros(sum(y_sep==cls)),
                   label=label, alpha=0.65, edgecolors='none')
axes[1].set_title('After PCA (1D)'); axes[1].set_xlabel('PC1'); axes[1].set_yticks([])
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_pca_vs_lda.png', dpi=150, bbox_inches='tight')
plt.show()

# Compare accuracy
model_orig = SVC(kernel='linear', random_state=42).fit(X_sep, y_sep)
model_pca  = SVC(kernel='linear', random_state=42).fit(X_pca1d, y_sep)
print(f'SVC accuracy — original 2D:  {model_orig.score(X_sep, y_sep):.3f}')
print(f'SVC accuracy — after PCA 1D: {model_pca.score(X_pca1d, y_sep):.3f}')

<a name="skip-code-12"></a>

**Expected output:** SVC accuracy drops substantially after PCA — often from above 0.95 to below 0.75. The scatter plots make the failure visible: the two classes are well separated in 2D but collapse onto each other in the 1D PCA projection.

**Accessibility note:** Add a Markdown cell describing both scatter plots. Note that the original 2D plot shows two distinct clusters, while the 1D PCA plot shows the classes overlapping on the single axis.

<nav aria-label="Return navigation for code cell 12">
<a href="#read-code-12" aria-label="Go back and read code cell 12">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-13"></a>

**Cell 13 — Implement LDA as the Correct Alternative**

Implement Linear Discriminant Analysis on the same dataset and compare accuracy. Add a comment explaining why LDA outperforms PCA here.

<nav aria-label="Code cell 13 navigation">
<a href="#skip-code-13" aria-label="Skip code cell 13 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# YOUR CODE HERE

# 1. Why did PCA fail?
# ANSWER:


# 2. Implement LDA and compare accuracy
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
# lda = ???
# X_lda = ???
# model_lda = ???
# print(f'SVC accuracy — after LDA 1D: {model_lda.score(X_lda, y_sep):.3f}')


# 3. When to use each method:
# Use PCA when: ???
# Use LDA when: ???
# Use t-SNE when: ???

<a name="skip-code-13"></a>

**Expected LDA accuracy:** Typically above 0.93 — much better than PCA.

**Solution summary:**

PCA failed because it finds directions of maximum *variance*, not maximum *class separation*. The principal component captured overall spread but projected both classes onto overlapping regions.

LDA is the correct alternative when class labels are available. It maximizes the ratio of between-class variance to within-class variance — explicitly optimizing for separability.

**When to use each:**
- PCA: unsupervised tasks, noise reduction, multicollinearity reduction, visualization without labels
- LDA: supervised classification, when labels are available and class separation matters
- t-SNE: visualization only — cannot reliably transform unseen data and is too slow for production

<nav aria-label="Return navigation for code cell 13">
<a href="#read-code-13" aria-label="Go back and read code cell 13">&#8617; Go back and read the code cell</a>
</nav>

---
# Part 4: Challenge Problems

These are timed interview pressure tests. **Set a timer before starting each one and work under interview conditions.** Narrate your reasoning as you code — in a real interview, silence is more costly than an imperfect answer.

---

## Challenge 4.1: The Feature Engineering Sprint
**Time Limit: 25 minutes**

**Scenario:** You are in a live coding interview. The interviewer says:
*'We have a subscriber churn dataset from a streaming platform. Our current model achieves 66% accuracy. Can you engineer features to improve it? You have 25 minutes.'*

**Rules:**
- Time limit: 25 minutes
- Must improve baseline accuracy
- Explain your reasoning as you code
- Be prepared to defend every feature you create

**Domain Context (streaming subscriber churn):**
Users cancel their subscription when they feel the platform no longer delivers value relative to cost. Key churn drivers: content engagement falling, price sensitivity, technical frustration, and lifecycle stage (new users and long-tenured users churn differently).

<a name="read-code-14"></a>

**Cell 14 — Generate the Streaming Churn Dataset and Establish a Baseline**

This cell creates a 600-subscriber dataset with features including account tenure, monthly streams watched, monthly cost, number of technical support tickets, content diversity score, device count, subscription tier, and days since last stream. A baseline Random Forest model is trained and its accuracy is printed. Your job is to beat this baseline in Cell 15.

<nav aria-label="Code cell 14 navigation">
<a href="#skip-code-14" aria-label="Skip code cell 14 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import time
start_time = time.time()

np.random.seed(42)
n = 600

subscriber_data = pd.DataFrame({
    'account_tenure_months':  np.random.randint(1, 72, n),
    'monthly_streams':        np.random.randint(0, 300, n),
    'monthly_cost_usd':       np.random.choice([8.99, 13.99, 17.99], n),
    'support_tickets_90d':    np.random.randint(0, 8, n),
    'content_diversity_score':np.random.uniform(0, 1, n),  # 0=narrow, 1=diverse
    'devices_registered':     np.random.randint(1, 6, n),
    'subscription_tier':      np.random.choice(['Basic','Standard','Premium'], n),
    'days_since_last_stream': np.random.randint(0, 60, n),
})

# Target: churn is driven by disengagement, frustration, and value perception
subscriber_data['churned'] = (
    (subscriber_data['days_since_last_stream'] > 30).astype(int) * 0.35
    + (subscriber_data['support_tickets_90d'] > 3).astype(int) * 0.25
    + (subscriber_data['monthly_streams'] < 10).astype(int) * 0.25
    + (subscriber_data['content_diversity_score'] < 0.2).astype(int) * 0.10
    + np.random.random(n) * 0.05
) > 0.45
subscriber_data['churned'] = subscriber_data['churned'].astype(int)

print('Dataset shape:', subscriber_data.shape)
print('\nChurn rate:')
print(subscriber_data['churned'].value_counts(normalize=True).round(3))

# Baseline model
X_base = subscriber_data.drop('churned', axis=1)
X_base = pd.get_dummies(X_base, drop_first=True)
y_churn = subscriber_data['churned']

X_tr, X_te, y_tr, y_te = train_test_split(X_base, y_churn, test_size=0.3, random_state=42)

baseline_model = RandomForestClassifier(n_estimators=100, random_state=42)
baseline_model.fit(X_tr, y_tr)
baseline_acc = baseline_model.score(X_te, y_te)

print(f'\nBASELINE ACCURACY: {baseline_acc:.3f}')
print('\nTimer started — 25 minutes.')
print('='*60)

<a name="skip-code-14"></a>

**Expected output:** Dataset shape (600, 9), churn rate around 35–45%, and a baseline accuracy around 0.65 to 0.70. Your engineered features in Cell 15 should push accuracy above baseline.

<nav aria-label="Return navigation for code cell 14">
<a href="#read-code-14" aria-label="Go back and read code cell 14">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-15"></a>

**Cell 15 — Feature Engineering (Your Work)**

Engineer features that beat the baseline model. Starter ideas are listed as comments inside the cell. For each feature you create, add a comment explaining what it represents and why you believe it helps predict churn. Build and evaluate your final model at the bottom of the cell.

<nav aria-label="Code cell 15 navigation">
<a href="#skip-code-15" aria-label="Skip code cell 15 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# FEATURE ENGINEERING — document your reasoning for each feature

eng = subscriber_data.copy()

# Starter ideas (expand and add your own):

# 1. Engagement rate: streams per dollar spent (value perception)
# eng['streams_per_dollar'] = eng['monthly_streams'] / (eng['monthly_cost_usd'] + 0.01)

# 2. Recency flag: user has not streamed in the last 2 weeks (leading indicator)
# eng['disengaged'] = (eng['days_since_last_stream'] > 14).astype(int)

# 3. Frustration index: support tickets relative to tenure (normalizes for older accounts)
# eng['frustration_rate'] = eng['support_tickets_90d'] / (eng['account_tenure_months'] + 1)

# 4. New subscriber flag: tenure < 3 months (early dropout risk)
# eng['new_subscriber'] = (eng['account_tenure_months'] < 3).astype(int)

# 5. Device engagement ratio: streams divided by devices (are devices actually being used?)
# eng['streams_per_device'] = eng['monthly_streams'] / eng['devices_registered']

# 6. Satisfaction proxy: diversity x streams (engaged AND exploring = healthy)
# eng['engagement_quality'] = eng['content_diversity_score'] * eng['monthly_streams']

# YOUR CODE HERE — add more features below


# --- Final model with engineered features ---
# X_eng = eng.drop('churned', axis=1)
# X_eng = pd.get_dummies(X_eng, drop_first=True)
# y_eng = eng['churned']
# X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(X_eng, y_eng, test_size=0.3, random_state=42)
# final_model = RandomForestClassifier(n_estimators=100, random_state=42)
# final_model.fit(X_tr_e, y_tr_e)
# final_acc = final_model.score(X_te_e, y_te_e)

elapsed = time.time() - start_time
print(f'Time elapsed: {elapsed/60:.1f} minutes')
print(f'Baseline accuracy: {baseline_acc:.3f}')
# print(f'Your accuracy:     {final_acc:.3f}')
# print(f'Improvement:       +{(final_acc - baseline_acc)*100:.1f}pp')

<a name="skip-code-15"></a>

**Evaluation guidance:**

A well-engineered set of features should push accuracy to 0.72 to 0.82. The strongest single features are usually `streams_per_dollar` (value perception), `disengaged` (recency), and `frustration_rate` (normalized support burden).

**Interview explanation template:**
'I engineered features based on known churn drivers: value perception (streams per dollar), engagement recency (days-since-stream flag), and normalized frustration (tickets per tenure month). These translate raw fields into signals more directly aligned with the outcome while remaining interpretable to the business team.'

<nav aria-label="Return navigation for code cell 15">
<a href="#read-code-15" aria-label="Go back and read code cell 15">&#8617; Go back and read the code cell</a>
</nav>

---
## Challenge 4.2: Dimensionality Reduction Decision
**Time Limit: 20 minutes**

**Scenario:** A live technical interview question:
*'We have a 64-feature handwritten digit recognition dataset. Recommend a dimensionality reduction approach, implement at least three methods, and justify your recommendation.'*

**Interview Conditions:**
- Explain your reasoning at each step
- Compare methods both visually and quantitatively
- Consider computational cost, interpretability, and whether the method generalizes to new data
- End with a clear, justified recommendation

<a name="read-code-16"></a>

**Cell 16 — Load the Digits Dataset and Set Up the Comparison**

This cell loads scikit-learn's digits dataset (1,797 images, 64 pixel features, 10 classes). It standardizes the features and splits into train and test sets. The setup printed at the end gives you the context to frame your recommendation.

<nav aria-label="Code cell 16 navigation">
<a href="#skip-code-16" aria-label="Skip code cell 16 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.datasets import load_digits
from sklearn.neighbors import KNeighborsClassifier
import time

digits    = load_digits()
X_dig     = digits.data
y_dig     = digits.target

print(f'Dataset: {X_dig.shape[0]} samples, {X_dig.shape[1]} features, {len(np.unique(y_dig))} classes')
print('Task: Reduce from 64D to 2D for visualization and classification')
print('\nYou must:')
print('1. Try at least 3 dimensionality reduction techniques')
print('2. Compare them visually and with a classifier')
print('3. Make a clear, justified recommendation')
print('\nTimer started — 20 minutes.')
print('='*60)

scaler_dig = StandardScaler()
X_dig_scaled = scaler_dig.fit_transform(X_dig)

X_d_tr, X_d_te, y_d_tr, y_d_te = train_test_split(
    X_dig_scaled, y_dig, test_size=0.3, random_state=42, stratify=y_dig
)

<a name="skip-code-16"></a>

**Expected output:** Dataset summary showing 1,797 samples, 64 features, 10 digit classes. The split gives you approximately 1,257 training samples and 540 test samples.

<nav aria-label="Return navigation for code cell 16">
<a href="#read-code-16" aria-label="Go back and read code cell 16">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-17"></a>

**Cell 17 — Implement and Compare Three Dimensionality Reduction Methods**

Implement PCA, LDA, and t-SNE, each reducing to 2 components. For each method: record the fit time, create a 2D scatter plot colored by digit class, and train a KNN classifier on the reduced training features to compute test accuracy. The starter structure is provided — fill in the blanks and add your comparison summary.

<nav aria-label="Code cell 17 navigation">
<a href="#skip-code-17" aria-label="Skip code cell 17 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from sklearn.manifold import TSNE

results = {}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# =====================================================================
# METHOD 1: PCA — Unsupervised, fast, generalizes to test data
# =====================================================================
t0 = time.time()
pca_2d   = PCA(n_components=2)
X_pca_tr = pca_2d.fit_transform(X_d_tr)
X_pca_te = pca_2d.transform(X_d_te)    # ✓ can transform test data
var_exp  = sum(pca_2d.explained_variance_ratio_)
t_pca    = time.time() - t0

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_pca_tr, y_d_tr)
acc_pca = knn.score(X_pca_te, y_d_te)
results['PCA'] = {'time': t_pca, 'accuracy': acc_pca, 'variance_explained': var_exp}

for cls in range(10):
    mask = y_d_tr == cls
    axes[0].scatter(X_pca_tr[mask, 0], X_pca_tr[mask, 1], label=str(cls), alpha=0.5, s=15)
axes[0].set_title(f'PCA (2D)\nAcc: {acc_pca:.3f} | Var: {var_exp:.1%}', fontsize=12)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].grid(alpha=0.3)

# =====================================================================
# METHOD 2: LDA — Supervised, maximizes class separation
# =====================================================================
# YOUR CODE HERE — follow the same pattern as PCA above
# lda_2d   = LDA(n_components=2)
# X_lda_tr = ???
# X_lda_te = ???
# knn.fit(X_lda_tr, y_d_tr)
# acc_lda  = knn.score(X_lda_te, y_d_te)
# results['LDA'] = {'time': ???, 'accuracy': acc_lda}
# (plot on axes[1])

# =====================================================================
# METHOD 3: t-SNE — Nonlinear visualization, training set only
# =====================================================================
# YOUR CODE HERE
# Note: t-SNE does not have a reliable transform() for test data
# t_tsne   = time.time()
# tsne     = TSNE(n_components=2, random_state=42, perplexity=30)
# X_tsne   = tsne.fit_transform(X_d_tr)
# results['tSNE'] = {'time': time.time()-t_tsne, 'accuracy': 'N/A (visualization only)'}
# (plot on axes[2])

plt.tight_layout()
plt.savefig('fig_dr_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Results Summary ===')
for method, r in results.items():
    print(f'{method:8s}: time={r["time"]:.2f}s | accuracy={r.get("accuracy","N/A")}')

<a name="skip-code-17"></a>

**Expected results (after completing the code):**

| Method | Time | KNN Accuracy | Notes |
| :--- | :--- | :--- | :--- |
| PCA | < 1s | ~0.92 | Unsupervised, generalizes |
| LDA | < 1s | ~0.95 | Supervised, best for classification |
| t-SNE | 5–15s | N/A | Visualization only, no test transform |

**Accessibility note:** Add a Markdown cell describing the three scatter plots. Note which shows clearer cluster separation and what that means for downstream classification.

**Your recommendation (write in the answer cell below):**
LDA is the right choice for this classification task — it is supervised, fast, can transform test data consistently, and produces the highest accuracy. Use t-SNE only for visualization and diagnostic insight, not as a production preprocessing step.

<nav aria-label="Return navigation for code cell 17">
<a href="#read-code-17" aria-label="Go back and read code cell 17">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-18"></a>

**Cell 18 — Write Your Recommendation**

Fill in your written recommendation as comments. Frame it as a spoken interview answer — clear, structured, and justified.

<nav aria-label="Code cell 18 navigation">
<a href="#skip-code-18" aria-label="Skip code cell 18 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# RECOMMENDATION (write as if speaking to an interviewer)

# My recommended method:
# ANSWER:

# Why not PCA for this task?
# ANSWER:

# Why not t-SNE for production?
# ANSWER:

# One-sentence interview summary:
# ANSWER:

<a name="skip-code-18"></a>

**Model recommendation:**

'I recommend LDA for this digit classification task because it is supervised — it uses the class labels to explicitly maximize the separation between the 10 digit classes. This results in higher downstream classifier accuracy than PCA while being just as fast. Critically, unlike t-SNE, LDA can reliably transform new test data using the same projection learned during training, making it suitable for a production pipeline. I would use t-SNE only as a diagnostic tool to visually confirm that the classes are separable before committing to the full pipeline.'

<nav aria-label="Return navigation for code cell 18">
<a href="#read-code-18" aria-label="Go back and read code cell 18">&#8617; Go back and read the code cell</a>
</nav>

---
# Congratulations!

You have completed the Module 4 interview simulation scenarios.

## What You Practiced

- **Warm-up:** Data leakage detection, the dummy variable trap, scaler and imputer selection
- **Intermediate:** Mutual Information vs Pearson Correlation, three-stage feature selection, PCA scree interpretation
- **Advanced:** Production preprocessing pipeline design, diagnosing PCA failure modes
- **Challenge:** Timed feature engineering with business framing, three-way dimensionality reduction comparison

## Interview Preparation Checklist

- Can you explain data leakage and how `fit` vs `transform` prevents it?
- Can you explain the dummy variable trap and when `drop_first` applies?
- Can you choose between Mean, Median, and Mode imputation from first principles?
- Can you choose between StandardScaler and RobustScaler based on data shape?
- Can you build a three-stage feature selection pipeline and justify each stage?
- Can you read a PCA scree plot and communicate the tradeoff to a non-technical stakeholder?
- Can you explain why PCA fails for classification and when to use LDA instead?
- Can you explain why t-SNE should not go into a production pipeline?
- Can you engineer interpretable domain features under time pressure?

## Common Interview Mistakes to Avoid

- Calling `fit_transform()` on test data
- Including all N dummy columns (the dummy variable trap)
- Choosing mean imputation for skewed or outlier-heavy features
- Forgetting to scale before PCA
- Using t-SNE as a production preprocessing step
- Selecting features on the full dataset before the train/test split (leakage)
- Reporting only accuracy when the classes are imbalanced

## Remember

> The interviewer cares more about your reasoning than perfect code. Explain your choices out loud as you work — silence costs more than an imperfect answer.

---

**To share:** Go to **Share** in Colab → set access to **Anyone with the link can view** → copy the link → paste into Canvas.